**`03_ingest_tiles`**

Script examples to import tiling polygons (e.g., for data downloads). A
precursor step, like admin referencing: run it for whatever tiling
schemes and admin scope your project needs (OpenBuildingMap, Landsat/
Sentinel, census block groups/tracts, ...), before ingesting entities
that depend on them.

# Configure

In [ ]:
import argparse

from openplaces.io.ingester import Ingester
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(description='Ingest tiles using a recipe')
parser.add_argument(
    '--recipe_id',
    help='Identifier of the recipe (e.g., "tile-osm-2025")',
)
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to ingest (e.g., "US-RI")',
    nargs='*',
)
parser.add_argument(
    '--reprocess',
    help='Reprocess input data from downloaded file',
    action='store_true',
)
parser.add_argument(
    '--redownload',
    help='Redownload input data from original source',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If True, print outputs while processing data',
    action='store_true',
)
parser.add_argument(
    '--keep_unzipped',
    help='If True, keeps unzipped datasets in heap folder after processing',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    # Tiles to download OpenBuildingMap
    '--recipe_id tile-obm-2025 '
    # Landsat and Sentinel satellite image tiles (with UTM projections)
    # '--recipe_id tile-landsat-2026 '
    # '--recipe_id tile-sentinel-2026 '
    # U.S. census block groups
    # '--recipe_id US_tile-census-2025_blockgroup '
    # U.S. census tracts
    # '--recipe_id US_tile-census-2025_tract '
    # Brunswick, NC (flood/hurricane risk case)
    # '--admin_ids US-NC-BR '
    '--admin_ids US '
    # Processing flags
    '--reprocess '
    # '--redownload '
    '--verbose '
    '--keep_unzipped '
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

In [ ]:
# Show recipe parameters
pretty_print(get_recipe_by_id(args.recipe_id))

# Ingest tile data

In [ ]:
ingester = Ingester(args.recipe_id, args.admin_ids, verbose=args.verbose)

In [ ]:
ingester.ingest(
    reprocess=args.reprocess,
    redownload=args.redownload,
    keep_unzipped=args.keep_unzipped,
)

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/.../'.
# If False, writes a test version of the script to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Inspect results

## Show full map

In [ ]:
ingester.show_ingested_geometries(fill=False, edgecolor='magenta')

## Show random tile with attributes

In [ ]:
ingester.show_random_entity()

# Link tiles
Tile-admin links (n:m `tile_id` <-> `admin_id` crosswalks) are saved
beside the tile output.

`get_entity_link_path` returns their filepath.

They are configured in the recipe's `entity_links` declarations.

The cell below loads and displays a sample of the saved link table.

In [ ]:
import pandas as pd

from openplaces.geo.link import get_entity_link_path

# Get the first admin recipe link declared in the tile recipe
links = ingester.recipe.get('entity_links') or []
if links:
    other_recipe_id = links[0]['recipe_id']
    link_path = get_entity_link_path(args.recipe_id, other_recipe_id)
    if link_path.exists():
        df = pd.read_parquet(link_path)
        print(f'Loaded link table from: {link_path.name}')
        display(df.head(10))
    else:
        print(f'Entity link not found at: {link_path}')
else:
    print('No entity links declared in recipe.')